In [1]:
from soh_core_en import *
from synth_en import *
from datetime import datetime, timedelta
import random

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

real_data_path='./data/real/'
sim_data_path='./data/simulated/'

real_results_path='./results/real/'
sim_results_path='./results/sim/'

In [2]:
EUCs=['B','E','M','P','S','V','Z']
EUCs_voltage=[132.5,133,83,134,124,82,60]

jmsh1507 = MOSFETParams(      # 150V
    r_ds_on_25c_total=0.052,  # R_ds(on) at 25°C (Ohms)
    temp_coeff_rel=0.01,      # Relative Thermal coefficient (1%/°C)
    r_wiring=0.0005           # Added resistance (shunts/busbars,Ohms)
)

hy5012w = MOSFETParams(       # 125V
    r_ds_on_25c_total=0.029,  # R_ds(on) at 25°C (Ohms)
    temp_coeff_rel=0.003,     # Relative Thermal coefficient (0.4%/°C)
    r_wiring=0.0005           # Added resistance (shunts/busbars,Ohms)
)

jmsh1504as = MOSFETParams(    # 150V
    r_ds_on_25c_total=0.040,  # R_ds(on) at 25°C (Ohms)
    temp_coeff_rel=0.005,     # Relative Thermal coefficient (0.5%/°C)
    r_wiring=0.0005           # Added resistance (shunts/busbars,Ohms)
)

lfpak56 = MOSFETParams(       # 80V
    r_ds_on_25c_total=0.030,  # R_ds(on) at 25°C (Ohms)
    temp_coeff_rel=0.006,     # Relative Thermal coefficient (0.6%/°C)
    r_wiring=0.0005           # Added resistance (shunts/busbars,Ohms)
)

EUCs_fets=[jmsh1507,jmsh1507,hy5012w,jmsh1507,jmsh1504as,jmsh1504as,lfpak56]

eucs_dict={}
for i in range(0,len(EUCs)):
    eucs_dict.update({EUCs[i]:(EUCs_voltage[i],EUCs_fets[i])})



# Real data Analysis

In [3]:
for euc in eucs_dict:
    euc_specs=eucs_dict.get(euc)
    wheel="Wheel_"+euc
    df_stats = analyze_folder_for_req(real_data_path+wheel,mosfet_params=euc_specs[1])
    
    df_alarms, thresholds = detect_alarms_gauss(df_stats,use_batt_metric=euc_specs[1] is not None)
    export_soh_pdf(df_stats,df_alarms,thresholds,wheel,real_results_path+wheel+'.pdf',True,force_gaussian_plots=True)
    df_alarms.to_csv(real_results_path+wheel+'_alarms.csv')
    

# Synthetic data generation

In [4]:
for euc in eucs_dict:
    euc_specs=eucs_dict.get(euc)
    wheel="Wheel_"+euc
    out_folder=wheel+"_bad"
    wheel_nominal_voltage=euc_specs[0]
    for mode in ['batt_only','mosfet_only', 'global']:
        try:
            print(out_folder,mode)
            generate_synthetic_folder(real_data_path+wheel,sim_data_path+out_folder+'_'+mode,wheel_nominal_voltage,
                                      years=5, wheel_name=wheel,mode=mode,knee_frac=0.95)
        except:
            print('Could not generate ageing data for wheel ',euc)

Wheel_B_bad batt_only
Wheel_B_bad mosfet_only
Wheel_B_bad global
Wheel_E_bad batt_only
Wheel_E_bad mosfet_only
Wheel_E_bad global
Wheel_M_bad batt_only
Wheel_M_bad mosfet_only
Wheel_M_bad global
Wheel_P_bad batt_only
Wheel_P_bad mosfet_only
Wheel_P_bad global
Wheel_S_bad batt_only
Wheel_S_bad mosfet_only
Wheel_S_bad global
Wheel_V_bad batt_only
Wheel_V_bad mosfet_only
Wheel_V_bad global
Wheel_Z_bad batt_only
Wheel_Z_bad mosfet_only
Wheel_Z_bad global


# Generating single outlier case

In [5]:
outlier_wheel='Wheel_S'
outlier_data_name='Wheel_S_bad_1st_outlier'

d=generate_synthetic_folder(real_data_path+'Wheel_S',sim_data_path+'Wheel_S_bad_1st_outlier',125,
                                      years=5, wheel_name='Wheel_S',mode='mosfet_only',knee_frac=0.999)

# Simulated ageing data analysis

In [6]:
for euc in eucs_dict:
    euc_specs=eucs_dict.get(euc)
    wheel="Wheel_"+euc+'_bad'

    for mode in ['batt_only','mosfet_only', 'global']:
        try:
            print(sim_data_path+wheel+'_'+mode)
            
            df_stats = analyze_folder_for_req(sim_data_path+wheel+'_'+mode,mosfet_params=euc_specs[1])
            
            df_alarms, thresholds = detect_alarms_gauss(df_stats,use_batt_metric=euc_specs[1] is not None)
            export_soh_pdf(df_stats,df_alarms,thresholds,wheel,sim_results_path+wheel+'_'+mode+'.pdf',True)
            df_alarms.to_csv(sim_results_path+wheel+'_'+mode+'_alarms.csv')
        except:
            print('Analyze failed for' , sim_data_path+wheel,mode)
        

./data/simulated/Wheel_B_bad_batt_only
./data/simulated/Wheel_B_bad_mosfet_only
./data/simulated/Wheel_B_bad_global
./data/simulated/Wheel_E_bad_batt_only
./data/simulated/Wheel_E_bad_mosfet_only
./data/simulated/Wheel_E_bad_global
./data/simulated/Wheel_M_bad_batt_only
./data/simulated/Wheel_M_bad_mosfet_only
./data/simulated/Wheel_M_bad_global
./data/simulated/Wheel_P_bad_batt_only
./data/simulated/Wheel_P_bad_mosfet_only
./data/simulated/Wheel_P_bad_global
./data/simulated/Wheel_S_bad_batt_only
./data/simulated/Wheel_S_bad_mosfet_only
./data/simulated/Wheel_S_bad_global
./data/simulated/Wheel_V_bad_batt_only
./data/simulated/Wheel_V_bad_mosfet_only
./data/simulated/Wheel_V_bad_global
./data/simulated/Wheel_Z_bad_batt_only
./data/simulated/Wheel_Z_bad_mosfet_only
./data/simulated/Wheel_Z_bad_global


# Analyzing single outlier sim

In [7]:
outlier_wheel='Wheel_S'
outlier_data_name='Wheel_S_bad_1st_outlier'

df_stats = analyze_folder_for_req(sim_data_path+outlier_data_name,mosfet_params=eucs_dict.get('S')[1])
            
df_alarms, thresholds = detect_alarms_gauss(df_stats,use_batt_metric=eucs_dict.get('S')[1] is not None)
export_soh_pdf(df_stats,df_alarms,thresholds,outlier_data_name,sim_results_path+outlier_data_name+'.pdf',True)
df_alarms.to_csv(sim_results_path+outlier_data_name+'_alarms.csv')